# 05 - Policy Analysis

Inference and qualitative analysis for the Mahjong transformer policy model.

Goals:
- inspect top-k predictions
- compare expert vs model actions
- analyze strategic behavior
- evaluate confidence and ambiguity

In [9]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.append(str(project_root))

In [10]:
import torch
import pandas as pd
import sqlite3
import gzip
import json

from src.data.tokenizer import MahjongTokenizer
from src.features.tensorization import Tensorizer
from src.data.collator import MahjongCollator

from src.models.transformer_model import (
    MahjongTransformer
)

In [11]:
from src.utils.tile_decoder import (
    tile136_to_string
)

from src.utils.constants import (
    ACTION_TYPES
)

In [12]:
from src.utils.action_decoder import (
    decode_action
)

In [13]:
from src.utils.state_renderer import (
    render_state
)

In [14]:
from src.analysis.shanten import (
    calculate_shanten,
    evaluate_discards
)

In [15]:
from src.analysis.ukeire import (
    evaluate_discards_ukeire
)

In [16]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [17]:
model = MahjongTransformer(
    embedding_dim=128,
    num_heads=8,
    num_layers=4,
    ff_dim=512,
).to(device)

/home/julia/miniforge3/envs/riichi-ai/lib/python3.11/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [18]:
checkpoint = torch.load(
    "../checkpoints/baseline_transformer/epoch_15.pt",

    map_location=device,
)

In [19]:
model.load_state_dict(
    checkpoint["model_state_dict"]
)

<All keys matched successfully>

In [20]:
model.eval()

MahjongTransformer(
  (embedding_layer): MahjongEmbedding(
    (token_type_embedding): Embedding(7, 128)
    (tile_embedding): Embedding(35, 128)
    (copy_embedding): Embedding(5, 128)
    (player_embedding): Embedding(5, 128)
    (action_type_embedding): Embedding(13, 128)
    (position_embedding): Embedding(512, 128)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1

In [21]:
tokenizer = MahjongTokenizer()

tensorizer = Tensorizer()

collator = MahjongCollator()

In [22]:
conn = sqlite3.connect(
    "../data/raw/datasets_positive.db"
)

query = """
SELECT Data
FROM Discard
LIMIT 1 OFFSET 250000
"""

row = conn.execute(query).fetchone()

In [23]:
decoded = json.loads(
    gzip.decompress(row[0])
)

In [24]:
canonical_state = {
    "round_wind": decoded["round_wind"],
    "num_honba": decoded["num_honba"],
    "num_riichi": decoded["num_riichi"],
    "player_wind": decoded["player_wind"],
    "position": decoded["position"],
    "remain_tiles": decoded["remain_tiles"],

    "dora_indicators": decoded["dora_indicators"],

    "hand_tiles": decoded["hand_tiles"],

    "players": {
        "0": decoded["0"],
        "1": decoded["1"],
        "2": decoded["2"],
        "3": decoded["3"],
    },

    "valid_actions": decoded["valid_actions"],

    "action_idx": decoded["action_idx"],
}

In [25]:
tokens = tokenizer.tokenize(
    canonical_state
)

tensor_dict = tensorizer.tensorize(
    tokens
)

batch = collator.collate([
    tensor_dict
])

In [26]:
moved = {}

for key, value in batch.items():

    if isinstance(value, list):

        moved[key] = [
            v.to(device)
            for v in value
        ]

    else:

        moved[key] = value.to(device)

batch = moved

In [27]:
with torch.no_grad():

    logits_list = model(batch)

/home/julia/miniforge3/envs/riichi-ai/lib/python3.11/site-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1739474893324/work/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


In [28]:
logits = logits_list[0]

probabilities = torch.softmax(
    logits,
    dim=0,
)

topk = torch.topk(
    probabilities,
    k=min(5, len(probabilities))
)

In [29]:
render_state(canonical_state)


ROUND

Round Wind: 0
Honba: 0
Riichi Sticks: 0
Remaining Tiles: 34

Dora Indicators: 8m

PLAYER 0

Points: 26900
Riichi: False
MELDS:
None

DISCARDS:
south green white 1m north* 8m 7s* east 2m* 5s*

PLAYER 1

Points: 17100
Riichi: False
MELDS:
None

DISCARDS:
north south 4s 3m 4s* green* east* north* 7m

PLAYER 2

Points: 34700
Riichi: False

HAND:
4m 4m 6p 6p 7p 1s 2s 3s 3s 4s 4s 5s 5s 7p

MELDS:
None

DISCARDS:
green 1p white green* 8m 4p west* 8s*

PLAYER 3

Points: 21300
Riichi: False
MELDS:
pon: ['white', 'white', 'white']

DISCARDS:
9s 1s south north 2s* red 6s west* 2m*


In [30]:
expert_idx = canonical_state["action_idx"]

expert_action = canonical_state[
    "valid_actions"
][expert_idx]

print("EXPERT ACTION:")

print(decode_action(expert_action))

EXPERT ACTION:
discard: ['1s']


In [31]:
print("\nMODEL TOP-5 ACTIONS:\n")

for rank in range(len(topk.indices)):

    action_idx = topk.indices[
        rank
    ].item()

    prob = topk.values[
        rank
    ].item()

    action = canonical_state[
        "valid_actions"
    ][action_idx]

    print(
        f"Rank {rank+1}"
    )

    print(
        f"Probability: {prob:.4f}"
    )

    print(
        f"Action Type: "
        f"{ACTION_TYPES[action['type']]}"
    )

    print(
        decode_action(action)
    )

    print()


MODEL TOP-5 ACTIONS:

Rank 1
Probability: 0.6923
Action Type: discard
discard: ['1s']

Rank 2
Probability: 0.2581
Action Type: discard
discard: ['2s']

Rank 3
Probability: 0.0186
Action Type: discard
discard: ['3s']

Rank 4
Probability: 0.0172
Action Type: discard
discard: ['3s']

Rank 5
Probability: 0.0038
Action Type: discard
discard: ['5s']



In [32]:
calculate_shanten(
    canonical_state["hand_tiles"]
)

0

In [33]:
results = evaluate_discards(
    canonical_state["hand_tiles"]
)

In [34]:
from src.utils.tile_decoder import (
    tile136_to_string
)

for r in results:

    print(
        tile136_to_string(
            r["discard"]
        ),
        "->",
        r["shanten"]
    )

4m -> 1
4m -> 1
6p -> 1
6p -> 1
7p -> 1
1s -> 0
2s -> 0
3s -> 1
3s -> 1
4s -> 1
4s -> 1
5s -> 1
5s -> 1
7p -> 1


In [35]:
results = evaluate_discards_ukeire(
    canonical_state["hand_tiles"]
)

In [36]:
from src.utils.tile_decoder import (
    tile136_to_string,
    tile34_to_string,
)

for r in results:

    improving = [
        tile34_to_string(t)
        for t in r["improving_tiles"]
    ]

    print()

    print(
        tile136_to_string(
            r["discard"]
        )
    )

    print(
        "Shanten:",
        r["shanten"]
    )

    print(
        "Ukeire:",
        r["ukeire"]
    )

    print(
        "Improving:",
        improving
    )


4m
Shanten: 1
Ukeire: 7
Improving: ['4m', '6p', '7p', '1s', '2s', '3s', '6s']

4m
Shanten: 1
Ukeire: 7
Improving: ['4m', '6p', '7p', '1s', '2s', '3s', '6s']

6p
Shanten: 1
Ukeire: 9
Improving: ['4m', '5p', '6p', '7p', '8p', '1s', '2s', '3s', '6s']

6p
Shanten: 1
Ukeire: 9
Improving: ['4m', '5p', '6p', '7p', '8p', '1s', '2s', '3s', '6s']

7p
Shanten: 1
Ukeire: 9
Improving: ['4m', '5p', '6p', '7p', '8p', '1s', '2s', '3s', '6s']

1s
Shanten: 0
Ukeire: 1
Improving: ['2s']

2s
Shanten: 0
Ukeire: 1
Improving: ['1s']

3s
Shanten: 1
Ukeire: 3
Improving: ['1s', '2s', '3s']

3s
Shanten: 1
Ukeire: 3
Improving: ['1s', '2s', '3s']

4s
Shanten: 1
Ukeire: 8
Improving: ['4m', '5p', '6p', '7p', '8p', '1s', '2s', '4s']

4s
Shanten: 1
Ukeire: 8
Improving: ['4m', '5p', '6p', '7p', '8p', '1s', '2s', '4s']

5s
Shanten: 1
Ukeire: 8
Improving: ['4m', '5p', '6p', '7p', '8p', '1s', '2s', '5s']

5s
Shanten: 1
Ukeire: 8
Improving: ['4m', '5p', '6p', '7p', '8p', '1s', '2s', '5s']

7p
Shanten: 1
Ukeire: 9
Improvin